# Phase 1 — Large PDF Extraction: CME Reference Demo

This notebook demonstrates the **dictionary-first** large-PDF extraction
pipeline end to end on the reference document
`data/input/Metals_Option_Products.pdf`.

It uses **only** the reusable services under `src/large_pdf_extractor/` and runs
with the deterministic **FakeLLMProvider** — no API keys or paid calls required.

The CME bulletin is a *reference example* of a difficult large PDF (repeated
headers/footers, dense tables, product sections, disclaimers). The pipeline
itself is generic and Finance-Market-Ops aware via configurable hints only.

## 1. Environment and path setup

In [1]:
import sys, os, json
from pathlib import Path

# Make the src/ package importable when running from the notebooks/ folder.
ROOT = Path.cwd()
if (ROOT / "src").exists():
    PROJECT_ROOT = ROOT
elif (ROOT.parent / "src").exists():
    PROJECT_ROOT = ROOT.parent
else:
    raise RuntimeError("Could not locate project root containing src/.")

SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print("Project root:", PROJECT_ROOT)
import large_pdf_extractor
print("large_pdf_extractor version:", large_pdf_extractor.__version__)

Project root: /home/user/largeInvoice
large_pdf_extractor version: 0.1.0


## 2. Validate that the reference input files exist

In [2]:
PDF_PATH = PROJECT_ROOT / "data" / "input" / "Metals_Option_Products.pdf"
RTF_PATH = PROJECT_ROOT / "data" / "input" / "metal_options_summary.rtf"
OUTPUT_DIR = PROJECT_ROOT / "data" / "output"

pdf_exists = PDF_PATH.exists()
rtf_exists = RTF_PATH.exists()
print(f"PDF present:  {pdf_exists}  ({PDF_PATH})")
print(f"RTF present:  {rtf_exists}  ({RTF_PATH})")

if not pdf_exists:
    print(
        "\nNOTE: The reference PDF is missing. Place it at the path above to run "
        "the full demo. The remaining cells require it."
    )

PDF present:  True  (/home/user/largeInvoice/data/input/Metals_Option_Products.pdf)
RTF present:  True  (/home/user/largeInvoice/data/input/metal_options_summary.rtf)


## 3. Inspect the PDF profile (PyMuPDF baseline)

In [3]:
from large_pdf_extractor.parsing import get_parser
from large_pdf_extractor.domain.models import ParseStrategy
from large_pdf_extractor.profiling.document_profiler import (
    DocumentProfiler, select_representative_chunks,
)
from large_pdf_extractor.chunking.chunker import ChunkingService

assert pdf_exists, "Reference PDF required for this cell."

parser = get_parser(ParseStrategy.PYMUPDF)
parsed = parser.parse(str(PDF_PATH))
profile = DocumentProfiler().profile(parsed)

print("Document id:           ", parsed.document_id)
print("Page count:            ", profile.page_count)
print("Total characters:      ", profile.total_char_count)
print("Low-text pages:        ", len(profile.empty_or_low_text_pages))
print("Detected table pages:  ", profile.detected_table_count)
print("Repeated headers:      ", profile.repeated_header_candidates[:3])
print("Repeated footers:      ", profile.repeated_footer_candidates[:3])
print("Sample headings:       ", profile.detected_headings[:5])

Document id:            Metals_Option_Products-89ddba8c4b
Page count:             60
Total characters:       766344
Low-text pages:         0
Detected table pages:   60
Repeated headers:       ['CME Group, Inc.', '2026 DAILY INFORMATION BULLETIN - http://www.cmegroup.com/dailybulletin', 'METALS OPTIONS PRODUCTS']
Repeated footers:       ['IS ACCEPTED BY THE USER ON THE CONDITION THAT ERRORS OR OMISSIONS SHALL NOT BE MADE THE BASIS FOR ANY CLAIM, DEMAND, OR CAUSE FOR ACTION.', '© Copyright CME Group Inc. All rights reserved.', 'THE INFORMATION CONTAINED IN THIS REPORT IS COMPILED FOR THE CONVENIENCE OF THE USER AND IS FURNISHED WITHOUT RESPONSIBILITY FOR ACCURACY OR CONTENT. IT']
Sample headings:        ['METALS OPTIONS PRODUCTS', 'METALS OPTION PRODUCTS', 'Side 01', 'PG64', 'Thu, May 28, 2026']


## 4. Dictionary proposal — BEFORE any extraction (dictionary-first)

In [4]:
from large_pdf_extractor.dictionary.generator import DictionaryService

chunks = ChunkingService().chunk(parsed, profile)
representative = select_representative_chunks(profile, chunks)

dictionary = DictionaryService().generate_proposed_dictionary(profile, representative)
print("Proposed dictionary:", dictionary.dictionary_id)
print("Item count:", len(dictionary.items))
for item in dictionary.items:
    print(f"  - {item.item_id:32s} [{item.expected_type.value}] section={item.document_section}")

Proposed dictionary: dict-Metals_Option_Products-89ddba8c4b
Item count: 10
  - document_title                   [string] section=document metadata
  - report_or_business_date          [date] section=document metadata
  - product_or_business_section      [string] section=product or business section
  - entity_identifiers               [list] section=entity identifiers
  - key_table_metrics                [table] section=key table metrics
  - operational_status               [list] section=operational status
  - exception_or_risk_indicators     [list] section=exception or risk indicators
  - totals_and_summary_values        [number] section=totals and summary values
  - disclaimers_or_caveats           [string] section=disclaimers or caveats
  - source_table_references          [list] section=source table references


## 5. Validate the dictionary with Pydantic

In [5]:
from large_pdf_extractor.dictionary.loader import DictionaryLoader
from large_pdf_extractor.domain.models import ExtractionDictionary

validated = DictionaryLoader().validate(dictionary.model_dump())
assert isinstance(validated, ExtractionDictionary)
assert validated.items, "Dictionary must contain items"
print("Dictionary validated OK with", len(validated.items), "items.")

Dictionary validated OK with 10 items.


## 6. PyMuPDF path — parse / profile / chunk / extract / render

We run the full pipeline through the orchestrator (the same code the CLI uses).

In [6]:
from large_pdf_extractor.app.config import build_run_config
from large_pdf_extractor.app.pipeline import Phase1Pipeline

py_config = build_run_config(
    pdf_path=str(PDF_PATH),
    output_dir=str(OUTPUT_DIR),
    strategy=ParseStrategy.PYMUPDF,
    llm_provider="fake",
    max_chunks=12,
)
py_state = Phase1Pipeline(py_config).run()
py_run_dir = OUTPUT_DIR / py_state.document_id / py_state.run_id
print("PyMuPDF run id:", py_state.run_id)
print("Artifacts written:", len(py_state.artifacts))
print("Run dir:", py_run_dir)

[05/29/26 20:36:35] INFO     Starting run run-20260529T203635Z for document Metals_Option_Products-89ddba8c4b      
                             (strategy=pymupdf, provider=fake)

PyMuPDF run id: run-20260529T203635Z
Artifacts written: 8
Run dir: /home/user/largeInvoice/data/output/Metals_Option_Products-89ddba8c4b/run-20260529T203635Z


## 7. Docling path — runs if installed, otherwise graceful skip

In [7]:
from large_pdf_extractor.parsing.docling_parser import DoclingParser

docling_available = DoclingParser().is_available()
print("Docling available:", docling_available)

dl_config = build_run_config(
    pdf_path=str(PDF_PATH),
    output_dir=str(OUTPUT_DIR),
    strategy=ParseStrategy.DOCLING,
    llm_provider="fake",
    max_chunks=12,
)
dl_state = Phase1Pipeline(dl_config).run()
dl_run_dir = OUTPUT_DIR / dl_state.document_id / dl_state.run_id

dl_parsed_meta = json.loads((dl_run_dir / "parsed_document.docling.json").read_text())["metadata"]
if dl_parsed_meta.get("skipped"):
    print("Docling path gracefully skipped:")
    for w in dl_parsed_meta.get("warnings", []):
        print("   warning:", w)
else:
    print("Docling parsed", dl_parsed_meta, "pages.")

Docling available: False


                    INFO     Starting run run-20260529T203635Z for document Metals_Option_Products-89ddba8c4b      
                             (strategy=docling, provider=fake)

Docling path gracefully skipped:


## 8. Compare mode — one shared dictionary across both parser paths

In [8]:
cmp_config = build_run_config(
    pdf_path=str(PDF_PATH),
    output_dir=str(OUTPUT_DIR),
    strategy=ParseStrategy.COMPARE,
    llm_provider="fake",
    max_chunks=12,
)
cmp_state = Phase1Pipeline(cmp_config).run()
cmp_run_dir = OUTPUT_DIR / cmp_state.document_id / cmp_state.run_id
print("Compare run id:", cmp_state.run_id)
print("Run dir:", cmp_run_dir)
print("Comparison report present:",
      (cmp_run_dir / "comparison_report.json").exists())

                    INFO     Starting run run-20260529T203635Z for document Metals_Option_Products-89ddba8c4b      
                             (strategy=compare, provider=fake)

Compare run id: run-20260529T203635Z
Run dir: /home/user/largeInvoice/data/output/Metals_Option_Products-89ddba8c4b/run-20260529T203635Z
Comparison report present: True


## 9. Generated JSON and Markdown artifacts

In [9]:
print("PyMuPDF run artifacts:")
for p in sorted(py_run_dir.iterdir()):
    print("  ", p.name)
print("\nCompare run artifacts:")
for p in sorted(cmp_run_dir.iterdir()):
    print("  ", p.name)

PyMuPDF run artifacts:
   chunks.docling.jsonl
   chunks.pymupdf.jsonl
   comparison_report.json
   comparison_report.md
   document_profile.docling.json
   document_profile.pymupdf.json
   extraction_dictionary.proposed.json
   extraction_dictionary.used.json
   extraction_result.docling.json
   extraction_result.docling.md
   extraction_result.pymupdf.json
   extraction_result.pymupdf.md
   parsed_document.docling.json
   parsed_document.pymupdf.json
   run_metadata.json

Compare run artifacts:
   chunks.docling.jsonl
   chunks.pymupdf.jsonl
   comparison_report.json
   comparison_report.md
   document_profile.docling.json
   document_profile.pymupdf.json
   extraction_dictionary.proposed.json
   extraction_dictionary.used.json
   extraction_result.docling.json
   extraction_result.docling.md
   extraction_result.pymupdf.json
   extraction_result.pymupdf.md
   parsed_document.docling.json
   parsed_document.pymupdf.json
   run_metadata.json


## 10. Reload and validate generated outputs through Pydantic

In [10]:
from large_pdf_extractor.domain.models import (
    ExtractionResult, ComparisonReport, ParsedDocument, DocumentProfile,
)

py_result = ExtractionResult.model_validate_json(
    (py_run_dir / "extraction_result.pymupdf.json").read_text()
)
cmp_report = ComparisonReport.model_validate_json(
    (cmp_run_dir / "comparison_report.json").read_text()
)
reloaded_doc = ParsedDocument.model_validate_json(
    (py_run_dir / "parsed_document.pymupdf.json").read_text()
)
reloaded_profile = DocumentProfile.model_validate_json(
    (py_run_dir / "document_profile.pymupdf.json").read_text()
)

populated = sum(1 for v in py_result.values if v.value is not None)
print(f"Extraction result reloaded: {populated}/{len(py_result.values)} values populated")
print("Comparison strategies:", [s.value for s in cmp_report.compared_strategies])
print("Parsed document pages:", reloaded_doc.page_count)
print("Profile table pages:", reloaded_profile.detected_table_count)

Extraction result reloaded: 7/10 values populated
Comparison strategies: ['pymupdf', 'docling']
Parsed document pages: 60
Profile table pages: 60


## 11. Markdown preview (extraction result + comparison report)

In [11]:
ext_md = (py_run_dir / "extraction_result.pymupdf.md").read_text()
cmp_md = (cmp_run_dir / "comparison_report.md").read_text()
print("===== extraction_result.pymupdf.md (first 1800 chars) =====\n")
print(ext_md[:1800])
print("\n\n===== comparison_report.md (first 1800 chars) =====\n")
print(cmp_md[:1800])

===== extraction_result.pymupdf.md (first 1800 chars) =====

# Extraction Result — `pymupdf`

- **Document ID:** `Metals_Option_Products-89ddba8c4b`
- **Dictionary ID:** `dict-Metals_Option_Products-89ddba8c4b`
- **Parser strategy:** `pymupdf`
- **Values populated:** 7 / 10

## Extracted Values

| Item | Entity | Value | Confidence | Source (pages / chunk) | Warnings |
|------|--------|-------|------------|------------------------|----------|
| `document_title` | document_title | METALS OPTION PRODUCTS | 0.50 | p1 (pymupdf-c0000-p1) |  |
| `report_or_business_date` | report_or_business_date | May 28, 2026 | 0.50 | p1 (pymupdf-c0000-p1) |  |
| `product_or_business_section` | product_or_business_section | METALS OPTION PRODUCTS | 0.50 | p1 (pymupdf-c0000-p1) |  |
| `entity_identifiers` | entity_identifiers | _null_ | 0.00 |  | value_not_found_in_candidates |
| `key_table_metrics` | key_table_metrics | THE CME GROUP DAILY BULLETIN ONLY DISPLAYS THOSE PRODUCTS/CONTRACT MONTHS THAT HAVE VOL

## 12. Artifact summary table

In [12]:
import pandas as pd

rows = []
for ref in cmp_state.artifacts:
    rows.append({
        "artifact_type": ref.artifact_type.value,
        "strategy": ref.strategy.value if ref.strategy else "-",
        "file": Path(ref.path).name,
    })
summary_df = pd.DataFrame(rows)
summary_df

,artifact_type,strategy,file
0,parsed_document,pymupdf,parsed_document.pymupdf.json
1,document_profile,pymupdf,document_profile.pymupdf.json
2,chunks,pymupdf,chunks.pymupdf.jsonl
3,parsed_document,docling,parsed_document.docling.json
4,document_profile,docling,document_profile.docling.json
5,chunks,docling,chunks.docling.jsonl
6,extraction_dictionary,-,extraction_dictionary.proposed.json
7,extraction_dictionary,-,extraction_dictionary.used.json
8,extraction_result,pymupdf,extraction_result.pymupdf.json
9,markdown_result,pymupdf,extraction_result.pymupdf.md


## 13. Final acceptance checklist

In [13]:
checks = {
    "Input PDF present": pdf_exists,
    "Profile computed (pages > 0)": profile.page_count > 0,
    "Dictionary proposed before extraction": len(dictionary.items) > 0,
    "Dictionary validated by Pydantic": isinstance(validated, ExtractionDictionary),
    "PyMuPDF extraction_result.json written": (py_run_dir / "extraction_result.pymupdf.json").exists(),
    "PyMuPDF extraction_result.md written": (py_run_dir / "extraction_result.pymupdf.md").exists(),
    "Docling path ran or skipped gracefully": (dl_run_dir / "parsed_document.docling.json").exists(),
    "Compare report JSON written": (cmp_run_dir / "comparison_report.json").exists(),
    "Compare report MD written": (cmp_run_dir / "comparison_report.md").exists(),
    "Shared dictionary used in compare": (cmp_run_dir / "extraction_dictionary.used.json").exists(),
    "Outputs reload via Pydantic": isinstance(py_result, ExtractionResult),
    "At least one value extracted with source span": any(
        v.source_spans for v in py_result.values
    ),
}

all_pass = all(checks.values())
for name, ok in checks.items():
    print(f"[{'PASS' if ok else 'FAIL'}] {name}")
print("\n" + ("ALL CHECKS PASSED ✅" if all_pass else "SOME CHECKS FAILED ❌"))
assert all_pass, "Phase 1 acceptance checklist failed."

[PASS] Input PDF present
[PASS] Profile computed (pages > 0)
[PASS] Dictionary proposed before extraction
[PASS] Dictionary validated by Pydantic
[PASS] PyMuPDF extraction_result.json written
[PASS] PyMuPDF extraction_result.md written
[PASS] Docling path ran or skipped gracefully
[PASS] Compare report JSON written
[PASS] Compare report MD written
[PASS] Shared dictionary used in compare
[PASS] Outputs reload via Pydantic
[PASS] At least one value extracted with source span

ALL CHECKS PASSED ✅
